<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Operaciones de Aprendizaje Automático III**
# **Tarea 1: Congelar RAG**

## **Objetivo**
**Detectar un fallo que el RAG no avisa y dejarlo cubierto**

Entender que el índice no es un archivo cualquiera, sino algo que depende del corpus, del modelo, de su versión y de cómo cortamos los documentos. La idea es medir cuánto se rompe cuando esas cuatro dejan de coincidir, y ponerle dos frenos para que la próxima vez se note.

## **Descripción**

Alguien cambia el modelo de embeddings y se olvida de reindexar, no salta ningún error y el generador rellena con lo que ya sabía: respuestas que suenan perfectas, armadas sobre los documentos equivocados. Es aca donde se rompe a propósito, se compara contra un set con respuestas conocidas, y se cierra con dos frenos: que el índice no se abra si no es el que corresponde, y un gate que no deje promover una corrida que empeoró.

## **Tareas**

Rellenar el código faltante (#TU CODIGO AQUI) para cumplir con las siguientes tareas:

1. Congelar el entorno: versiones de librerías y revisiones exactas de cada modelo del Hub.
2. Darle identidad al índice, con un **index_id** que dependa del corpus, el modelo, su revisión
   y la política de chunking.
3. Medir la degradación con controles: cuánto se rompe el sistema al consultar un índice con un
   modelo que no lo construyó, usando una condición de control y un piso aleatorio.
4. Mostrar que el generador tapa el fallo, con un caso donde el retriever pierde el documento
   correcto y la respuesta suena igual de bien
5. Registrar todo en MLflow: parámetros, métricas y artefactos.
6. Convertir el fallo silencioso en un error que se note, con una validación al cargar el índice y
   un gate que compare contra un baseline registrado.

**Se entrega el cuaderno ejecutado en el formulario: [https://forms.gle/97XDqsezFFGYYkML8](https://forms.gle/97XDqsezFFGYYkML8) hasta el 01 de octubre a horas 23:59 por 10 puntos, después cada semana baja 1 punto**

### **a) Entorno**


In [ ]:
!pip -q install -U "transformers==4.51.3" "accelerate==1.6.0" "mlflow==2.21.3" "sentence-transformers==5.7.0"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import torch, json, re, os, gc, time, random, hashlib
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from huggingface_hub import list_repo_commits
from mlflow.tracking import MlflowClient
import mlflow
from importlib.metadata import version

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)

versiones = {p: version(p) for p in ["transformers", "accelerate", "mlflow",
                                     "sentence-transformers", "torch", "numpy"]}

hay_gpu = torch.cuda.is_available()
gpu_nombre = torch.cuda.get_device_name(0) if hay_gpu else "no hay gpu"
# bf16 pide Ampere o más, en la T4 no queda otra que fp16
dtype = torch.bfloat16 if hay_gpu and torch.cuda.get_device_properties(0).major >= 8 else torch.float16

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("clase3-congelar-rag")

def cerrar_runs():
    while mlflow.active_run() is not None:
        mlflow.end_run()

cerrar_runs()
print("gpu:", gpu_nombre, "| dtype:", dtype)
print(versiones)

### **b) Corpus y set de evaluación**

* El corpus debe ser difícil, si cada consulta es una paráfrasis de su único chunk, **recall@k** da 1.000 en todas las condiciones y la métrica no discrimina nada, por eso está armado en familias de chunks casi idénticos que difieren en un detalle.

In [ ]:
CORPUS = [
    # devoluciones, cambia el producto
    "El plazo de devolución de productos de electrónica es de 15 días corridos.", # 0
    "El plazo de devolución de indumentaria es de 30 días corridos.", # 1
    "El plazo de devolución de artículos de bazar es de 45 días corridos.", # 2
    "El plazo de devolución de libros y papelería es de 10 días corridos.", # 3

    # reembolsos, cambia el medio de pago
    "El reembolso por transferencia bancaria se acredita en 5 días hábiles.", # 4
    "El reembolso a tarjeta de crédito se acredita en 10 días hábiles.", # 5
    "El reembolso como crédito en tienda queda disponible de forma inmediata.", # 6

    # facturación, cambia el plan
    "El plan Premium se factura por mes adelantado, los días 1 de cada mes.", # 7
    "El plan Business se factura por mes vencido, los días 10 de cada mes.",# 8
    "El plan Básico se factura trimestralmente y no admite prorrateo.",# 9

    # suspensión, cambia la causa
    "Si el pago con tarjeta es rechazado, el servicio se suspende a las 72 horas.", # 10
    "Si la transferencia no se acredita, el servicio se suspende a los 5 días.", # 11

    # baja de cuenta, forzada o voluntaria
    "Tras la suspensión por falta de pago, la cuenta se elimina a los 30 días.", # 12
    "Tras una baja voluntaria, la cuenta se conserva 180 días antes de borrarse.", # 13

    # cambios de plan, cambia la dirección
    "El upgrade de plan a mitad de mes prorratea el saldo a favor del usuario.", # 14
    "El downgrade de plan recién impacta en el siguiente ciclo de facturación.",   # 15

    # atención, cambia el área
    "El soporte técnico atiende de lunes a viernes de 9 a 18, hora local.", # 16
    "El soporte comercial atiende de lunes a sábado de 10 a 20, hora local.", # 17

    # reclamos, cambia el motivo
    "Los reclamos por facturación se responden dentro de las 48 horas hábiles.", # 18
    "Los reclamos por garantía se responden dentro de los 5 días hábiles.",# 19

    # garantías, cambia el alcance
    "La garantía legal cubre 6 meses desde la fecha de compra.", # 20
    "La garantía extendida cubre 12 meses adicionales sobre la garantía legal.", # 21
    "La garantía no cubre daños por mal uso, golpes ni desgaste normal.", # 22

    # envíos, cambia el destino
    "Los envíos a CABA y GBA demoran 24 horas hábiles.", # 23
    "Los envíos al interior del país demoran entre 3 y 7 días hábiles.", # 24

    # cancelación de envío, antes o después del despacho
    "La cancelación del envío antes del despacho no tiene ningún costo.",# 25
    "La cancelación del envío una vez despachado tiene un cargo del 10% del total.", # 26

    # factura, cambia la condición fiscal
    "La factura A se emite solo con CUIT y condición de IVA declarada.",# 27
    "La factura B se emite por defecto a consumidor final, sin datos fiscales.", # 28

    # medios de pago, cambia el beneficio
    "El pago con tarjeta de crédito admite hasta 12 cuotas sin interés.",# 29
    "El pago por transferencia es en un solo pago y tiene 5% de descuento.",# 30

    # datos de la cuenta, cambia el campo
    "El cambio de correo electrónico requiere verificación y demora 24 horas.", # 31
    "El cambio de número de teléfono impacta de forma inmediata.",# 32

    # cupones, cambia con qué se combinan
    "El cupón de descuento no es acumulable con promociones bancarias.",# 33
    "El cupón de descuento sí es acumulable con el beneficio de envío gratis.", # 34

    # retiro en sucursal, cambia el momento
    "El pedido para retirar en sucursal queda disponible a las 48 horas.",# 35
    "El pedido en sucursal se guarda 7 días corridos antes de volver al depósito.", # 36

    # datos personales, cambia el trámite
    "La eliminación de datos personales a pedido del titular se completa en 30 días.",# 37
    "La exportación de datos personales se entrega dentro de las 72 horas.", # 38

    # cambios de talle, cambia cuál cambio es
    "El primer cambio de talle no tiene cargo de envío.",# 39
    "El segundo cambio de talle en adelante tiene cargo de envío a cargo del cliente.",# 40
]

# cada consulta obliga a elegir dentro de una familia de chunks parecidos
PARES = [
    ("¿cuántos días tengo para devolver un televisor?", 0),
    ("¿cuánto tiempo tengo para devolver una remera?", 1),
    ("compré una olla y no me gustó, ¿hasta cuándo la puedo devolver?", 2),
    ("¿hasta cuándo puedo devolver un libro?", 3),
    ("¿cuándo me devuelven el dinero si pagué con tarjeta?", 5),
    ("¿y si pedí la devolución por transferencia?", 4),
    ("si me dan crédito en la tienda, ¿cuándo lo puedo usar?", 6),
    ("¿qué día me cobran si tengo el plan business?", 8),
    ("tengo el plan premium, ¿me cobran antes o después del mes?", 7),
    ("¿el plan básico me devuelve la diferencia si me doy de baja antes?", 9),
    ("¿qué pasa si me rechazan la tarjeta?", 10),
    ("hice la transferencia y todavía no figura, ¿cuándo me cortan el servicio?", 11),
    ("si me doy de baja yo, ¿cuánto tiempo guardan mi cuenta?", 13),
    ("si me suspenden por no pagar, ¿cuándo se borra todo?", 12),
    ("si paso a un plan más caro a mitad de mes, ¿me cobran el mes entero?", 14),
    ("si bajo de plan, ¿desde cuándo me sale más barato?", 15),
    ("¿hasta qué hora atiende el área comercial?", 17),
    ("¿el soporte técnico atiende los sábados?", 16),
    ("¿cuánto tardan en contestar un reclamo de garantía?", 19),
    ("puse un reclamo porque me facturaron de más, ¿cuándo me responden?", 18),
    ("se me cayó el producto, ¿la garantía lo cubre?", 22),
    ("¿cuánto dura la garantía que viene de fábrica?", 20),
    ("si pago la garantía extendida, ¿cuánto tiempo más tengo?", 21),
    ("¿cuánto demora un envío a Córdoba?", 24),
    ("vivo en Palermo, ¿cuándo me llega el pedido?", 23),
    ("ya lo despacharon y me arrepentí, ¿me cobran algo por cancelar?", 26),
    ("todavía no salió el paquete, ¿puedo cancelar sin cargo?", 25),
    ("tengo CUIT, ¿qué necesitan para hacerme factura A?", 27),
    ("no tengo empresa, ¿qué tipo de factura me mandan?", 28),
    ("¿puedo pagar en cuotas con tarjeta?", 29),
    ("¿hacen descuento si pago por transferencia?", 30),
    ("quiero cambiar el correo de mi cuenta, ¿es inmediato?", 31),
    ("cambié de número de celular, ¿cuándo se actualiza?", 32),
    ("tengo un cupón, ¿lo puedo usar junto con la promoción del banco?", 33),
    ("¿el cupón se puede combinar con el envío gratis?", 34),
    ("compré para retirar en sucursal, ¿cuándo lo puedo ir a buscar?", 35),
    ("¿cuántos días guardan el pedido en la sucursal?", 36),
    ("pedí que borren mis datos, ¿cuánto tardan?", 37),
    ("quiero una copia de mis datos, ¿en cuánto me la mandan?", 38),
    ("me quedó grande la campera, ¿el primer cambio tiene costo?", 39),
    ("ya cambié una vez y me quedó mal de nuevo, ¿tengo que pagar?", 40),
]

K = 3   # con distractores
consultas = [q for q, _ in PARES]
gold = [g for _, g in PARES]
assert len(set(consultas)) == len(PARES) and all(0 <= g < len(CORPUS) for g in gold)

piso = K / len(CORPUS) # recall@K de un ranking al azar
corpus_hash = hashlib.sha256("||".join(CORPUS).encode()).hexdigest()[:12]
print(f"corpus {len(CORPUS)} chunks , evaluación {len(PARES)} consultas , K={K}")
print(f"resolución de recall@{K}: {100/len(PARES):.1f}%  ,  piso aleatorio: {piso:.3f}")
print("corpus_hash:", corpus_hash)

### **c) pinneamos revisiones**

* **from_pretrained("algun/modelo")** apunta a main, que es una rama y se puede mover
* la primera corrida resuelve main una vez y lo guarda en *+revisiones.json**, de ahí en adelante manda el archivo, que en producción se versiona junto con el código
* usamos dos modelos distintos en lugar de dos revisiones del mismo porque los repos populares casi nunca cambian sus pesos, su historial son READMEs y exports, y la degradación daría 0%.
* los dos descienden de XLM-RoBERTa-large y los dos están entrenados para recuperación asi que comparten tokenizer, vocabulario y parte de la geometría de su espacio, por eso el cruce degrada sin colapsar, que es lo que se ve en la práctica cuando alguien migra a la versión siguiente de la misma familia
* la dimensión de 1024 en los dos tiene su razón, con dimensiones distintas
numpy tiraría un error de shape y el fallo se vería enseguida


In [ ]:
GEN = "Qwen/Qwen2.5-3B-Instruct"
EMB_A = "BAAI/bge-m3" # 1024 dims, no usa prefijos
EMB_B = "intfloat/multilingual-e5-large"# 1024 dims, usa prefijos

if os.path.exists("revisiones.json"):
    revs = json.load(open("revisiones.json"))
    print("revisiones leídas del archivo, si el hub cambio no nos dimos cuenta")

else:
    revs = {r: list_repo_commits(r)[0].commit_id for r in (GEN, EMB_A, EMB_B)}
    json.dump(revs, open("revisiones.json", "w"), indent=2)
    print("revisiones resueltas desde main y congeladas")

for r, c in revs.items():
    print(f"{r}:{c}")

### **d) los modelos**

* cada embedder tiene un contrato de uso, acá son los prefijos, está en el READMEy si lo ignoramos el modelo no falla pero funciona pero mal
* lo escribimos como dato para poder versionarlo y registrarlo
* El índice sale del corpus, el modelo, su revisión y la política de chunking. * Si cambia cualquiera de los cuatro deja de ser válido
* un hash de esos cuatro alcanza para poder verificar después que el índice sea el que corresponde

In [ ]:
t0 = time.time()
dispositivo = "cuda" if #TU CODIGO AQUI else "cpu"
emb = {r: SentenceTransformer(r, revision=revs[r], device=dispositivo) for r in (EMB_A, EMB_B)}

PREFIJOS = {EMB_A: {"consulta": "", "pasaje": ""},
            EMB_B: {"consulta": "query: ", "pasaje": "passage: "}}

def dim(m):
    f = getattr(m, "get_embedding_dimension", None) or m.get_sentence_embedding_dimension
    return int(f())

assert dim(emb[EMB_A]) == dim(emb[EMB_B]), "dimensiones distintas, el fallo sería ruidoso"
print(f"embedders cargados en {dispositivo}, {time.time()-t0:.0f}s\n")
print(f"A {EMB_A} {dim(emb[EMB_A])} dims  {PREFIJOS[EMB_A]}")
print(f"B {EMB_B} {dim(emb[EMB_B])} dims  {PREFIJOS[EMB_B]}")

es la misma dimensión, numpy no da error, el fallo es silencioso

este corpus no se chunkea ya que cada entrada ya es un chunk, pero lo declaramos igual porque cambiar la política cambia el índice

In [ ]:
CHUNKING = {"estrategia": "uno-por-documento", "size": None, "overlap": None}

def index_id(corpus_hash, repo, rev, chunking):
    s = f"{corpus_hash}|{repo}@{rev}|chunk={json.dumps(#TU CODIGO AQUI, sort_keys=True)}"
    return hashlib.sha256(s.encode()).hexdigest()[:12]

def indexar(repo, chunks):
    # normalizados, así el producto punto ya es el coseno
    v = emb[repo].encode([PREFIJOS[repo]["pasaje"] + c for c in #TU CODIGO AQUI],
                         normalize_embeddings=#TU CODIGO AQUI, show_progress_bar=False, batch_size=16)
    return np.asarray(v, dtype=np.float32)

def encodear(repo, consultas, con_prefijo=True):
    p = PREFIJOS[repo]["consulta"] if con_prefijo else ""
    v = emb[repo].encode([p + q for q in consultas], normalize_embeddings=True,
                         show_progress_bar=False, batch_size=16)
    return np.asarray(v, dtype=np.float32)

def buscar(indice, vec, k=K):
    puntajes = indice @ np.asarray(vec, dtype=np.float32)
    top = np.argsort(-puntajes)[:k]
    return top.tolist(), puntajes[top].tolist()

def guardar(v, repo, nombre):
    # el npy solo es un array, sin el meta al lado nadie sabe con qué modelo salió
    iid = index_id(#TU CODIGO AQUI, #TU CODIGO AQUI, revs[#TU CODIGO AQUI], #TU CODIGO AQUI)
    np.save(f"{nombre}.npy", v)
    json.dump({"index_id": iid, "emb_repo": repo, "emb_revision": revs[repo],
               "chunking": CHUNKING, "corpus_hash": corpus_hash, "dim": int(v.shape[1])},
              open(f"{nombre}.meta.json", "w"), indent=2)
    return iid

t0 = time.time()
indice = {r: indexar(r, CORPUS) for r in (EMB_A, EMB_B)}
iid = {r: guardar(indice[r], r, "indice_" + ("a" if r == EMB_A else "b")) for r in (EMB_A, EMB_B)}
for r in (EMB_A, EMB_B):
    print(f"{r} shape {indice[r].shape}  index_id {iid[r]}")
print(f"\nindexado en {time.time()-t0:.0f}s")

### **e) experimento**

Cinco condiciones:

* A_indiceA: grupo coherente, es el baseline
* B_indiceB: grupo coherente, es el techo de B
* B_indiceA_sinpref: grupo cruzado, es el caso realista, solo se migró el modelo
* B_indiceA_conpref: grupo cruzado, es el B, pero sobre un indice ajeno
* aleatorio: grupo piso, es el suelo exacto

Comparando **A_indiceA** contra **B_indiceA** no se puede
separar si el índice es incompatible o si B es peor retriever en este corpus.

Las dos cruzadas separan otra cosa, si B con prefijo también degrada, el problema no era el
prefijo sino el índice.

Al final medimos si un umbral sobre el puntaje habría servido de alerta, que suele ser lo
primero que se intenta, y después miramos qué es lo que ese umbral separa.

In [ ]:
# nombre, grupo, repo de la consulta, repo del índice, usar prefijo
CONDICIONES = [
    ("A_indiceA", "coherente", #TU CODIGO AQUI, #TU CODIGO AQUI, True),
    ("B_indiceB", "coherente", #TU CODIGO AQUI, #TU CODIGO AQUI, True),
    ("B_indiceA_sinpref", "cruzado", #TU CODIGO AQUI, #TU CODIGO AQUI, False),
    ("B_indiceA_conpref", "cruzado",#TU CODIGO AQUI, #TU CODIGO AQUI, True),
    ("aleatorio", "piso", None,EMB_A, False),
]

vecs = {(EMB_A, True): encodear(EMB_A, consultas, True),
       (EMB_B, True): encodear(EMB_B, consultas, True),
       (EMB_B, False): encodear(EMB_B, consultas, False)}

recuperados, puntajes = {}, {}
for nombre, grupo, qrepo, irepo, pref in CONDICIONES:
    ids, pts = [], []
    if qrepo is None:
        rnd = random.Random(SEMILLA)
        for _ in consultas:
            ids.append(rnd.sample(range(len(CORPUS)), K))
            pts.append([float("nan")] * K)
    else:
        for j in range(len(consultas)):
            i, p = buscar(indice[irepo], vecs[(qrepo, pref)][j])
            ids.append(#TU CODIGO AQUI)
            pts.append(#TU CODIGO AQUI)
    recuperados[nombre], puntajes[nombre] = ids, pts

ninguna excepción, ningún warning, K documentos devueltos en las cinco condiciones desde afuera el sistema está bien


In [ ]:
for nombre, grupo, qrepo, _, _ in CONDICIONES:
    if qrepo:
        print(f"{nombre} puntaje top-1 medio = {np.mean([p[0] for p in puntajes[nombre]]):.3f}")

comparamos B contra sí mismo, los mismos vectores de consulta sobre su índice y sobre el de A

In [ ]:
sanos = [p[0] for p in puntajes["B_indiceB"]]
rotos = [p[0] for p in puntajes["B_indiceA_conpref"]]
cortes = sorted(set(sanos + rotos))

el umbral se elige con las etiquetas puestas, que en producción no se tienen, así que esto es un techo de lo que un umbral podría lograr

In [ ]:
exact = [(sum(1 for s in sanos if s >= t) + sum(1 for r in rotos if r < t))
         / (len(sanos) + len(rotos)) for t in cortes]
i = max(range(len(cortes)), key=lambda k: exact[k])

print(f"puntajes top-1 sanos: {min(sanos):.3f} a {max(sanos):.3f}")
print(f"puntajes top-1 rotos: {min(rotos):.3f} a {max(rotos):.3f}")
print(f"mejor umbral posible: {cortes[i]:.3f},  exactitud {exact[i]:.0%}\n")
if exact[i] < 0.85:
    print("ni el mejor umbral separa bien")
else:
    print("el umbral separa, pero ojo con lo que separa: cada modelo tiene su propia escala de similitud, así que lo que se está detectando es que el índice es de otro modelo, no que la recuperación esté mal")

### **f) Las métricas**

* **recall@k** revisa si el chunk correcto está en el top-k
* **hit@1** revisa si salió primero y es más sensible
* **mrr@k** mira en qué posición salió y capta el reordenamiento
* **overlap@k** es la que más importa en un RAG, al generador no le llega el chunk correcto, le llegan los k chunks, así que si el correcto sale primero en las dos condiciones pero los otros k-1 cambiaron, **recall@k** da 1.000 en las dos y el modelo está leyendo otra cosa.

El bootstrap remuestrea las consultas con reemplazo y mide cuánto se mueve el promedio

In [ ]:
def medir(ids, ref):
    rec, hit, mrr, ovl = [], [], [], []
    for j, g in enumerate(gold):
        top = ids[j][:K]
        rec.append(1.0 if g in top else 0.0)
        hit.append(1.0 if top[0] == g else 0.0)
        mrr.append(1.0 / (top.index(g) + 1) if g in top else 0.0)
        ovl.append(len(set(ref[j][:K]) & set(top)) / K)
    return {"recall": rec, "hit1": hit, "mrr": mrr, "overlap": ovl}

def ic_bootstrap(xs, n=1000):
    rnd = random.Random(SEMILLA)
    medias = sorted(sum(rnd.choices(xs, k=len(xs))) / len(xs) for _ in range(n))
    return medias[int(0.025 * n)], medias[int(0.975 * n)]

ref = recuperados["A_indiceA"]
met, ic = {}, {}
for nombre, *_ in CONDICIONES:
    pc = medir(recuperados[nombre], ref)
    met[nombre] = {k: sum(v) / len(v) for k, v in pc.items()}
    ic[nombre] = {k: ic_bootstrap(v) for k, v in pc.items()}

print(f"{'condición':<20}{'recall@3 [IC95]':<24}{'overlap@3 [IC95]':<24}{'hit@1':>7}{'mrr':>7}")
print("-" * 82)
for nombre, *_ in CONDICIONES:
    cr = f"{met[nombre]['recall']:.3f} [{ic[nombre]['recall'][0]:.2f}-{ic[nombre]['recall'][1]:.2f}]"
    co = f"{met[nombre]['overlap']:.3f} [{ic[nombre]['overlap'][0]:.2f}-{ic[nombre]['overlap'][1]:.2f}]"
    print(f"{nombre:<20}{cr:<24}{co:<24}{met[nombre]['hit1']:>7.3f}{met[nombre]['mrr']:>7.3f}")
print("-" * 82)
print(f"piso teórico de recall@{K}: {piso:.3f}\n")

# cada cruzada contra su propio techo, no contra A
techo = {EMB_A: "A_indiceA", EMB_B: "B_indiceB"}
print("costo de no reindexar")
for nombre, grupo, qrepo, irepo, pref in CONDICIONES:
    if grupo != "cruzado":
        continue
    t = techo[qrepo]
    base, cand = met[t]["recall"], met[nombre]["recall"]
    col = f"{cand:.3f} vs {base:.3f} ({(cand-base)/base*100:+.0f}%)"
    print(f"  {nombre:<20}techo {t:<12}{col:<26}overlap {met[nombre]['overlap']:.3f}")

* Para leer la tabla cada cruzada se compara contra su propio techo y no contra A, así la resta
es el costo de no reindexar sin mezclarlo con qué tan bueno es B.
* **overlap** da más información que **recall**, que tiene efecto techo.
* El piso sirve de calibración: una cruzada bastante por encima del azar quiere decir que el fallo es parcial y un fallo parcial es más difícil de notar
que uno total, y si dos intervalos se superponen, la diferencia entre esas dos condiciones no está establecida.

### **g) El registro en MLflow**

Un run por condición, el **index_id** va como param porque es una entrada y **overlap** va como metric porque es un resultado. Si dos runs difieren en una métrica, la causa
está en alguno de los params

In [ ]:
cerrar_runs()
runs = {}
for nombre, grupo, qrepo, irepo, pref in CONDICIONES:
    with mlflow.start_run(run_name=nombre) as run:
        mlflow.log_params({"grupo": #TU CODIGO AQUI,
                           "repo_consulta": #TU CODIGO AQUI or "ninguno",
                           "repo_indice": #TU CODIGO AQUI,
                           "revision_indice": #TU CODIGO AQUI,
                           "index_id": #TU CODIGO AQUI,
                           "con_prefijo": #TU CODIGO AQUI,
                           "k": #TU CODIGO AQUI,
                           "corpus_hash": #TU CODIGO AQUI,
                           "chunking": json.dumps(#TU CODIGO AQUI, sort_keys=True),
                           "gpu": #TU CODIGO AQUI, "dtype": str(dtype), "semilla": #TU CODIGO AQUI,
                           **{f"v_{p}": v for p, v in versiones.items()}})

        # mlflow no acepta "@" en el nombre de una métrica, por eso recall y no recall@3
        mlflow.log_metrics(met[nombre])
        mlflow.log_metrics({f"{k}_ic_lo": ic[nombre][k][0] for k in met[nombre]})
        mlflow.log_metrics({f"{k}_ic_hi": ic[nombre][k][1] for k in met[nombre]})
        runs[nombre] = run.info.run_id

with mlflow.start_run(run_name="artefactos"):
    for f in ["revisiones.json", "indice_a.npy", "indice_a.meta.json",
              "indice_b.npy", "indice_b.meta.json"]:
        mlflow.log_artifact(f)

print(f"{len(runs)} runs registrados")

### **h) El generador**

* En la sección j le vamos a pedir que tome en cuenta un contexto que contradice lo que el modelo ya sabe y los modelos chicos pequeños contestan de memoria. Si pasa eso, el experimento termina midiendo al generador en lugar del retriever.

* **responder** recibe ids fijos y no busca nada, así se varía una cosa por vez.
* La decodificación es greedy y apagamos **temperature**, **top_p** y **top_k** a mano, porque el **generation_config.json** del repo trae **do_sample=True** y el sampling lo terminaba decidiendo ese archivo

In [ ]:
if "gen" in globals():
    del gen
gc.collect()
if hay_gpu:
    torch.cuda.empty_cache()

t0 = time.time()
tok = AutoTokenizer.from_pretrained(#TU CODIGO AQUI, revision=revs[#TU CODIGO AQUI])

# device_map fijo en cuda:0, con "auto" a veces manda capas a CPU o disco sin avisar
gen = AutoModelForCausalLM.from_pretrained(#TU CODIGO AQUI, revision=revs[#TU CODIGO AQUI], torch_dtype=dtype,
                                           device_map={"": 0} if hay_gpu else None)
print(f"cargado en {time.time()-t0:.0f}s | footprint {gen.get_memory_footprint()/1e9:.2f} GB")
if hay_gpu:
    print(f"VRAM ocupada: {torch.cuda.memory_allocated()/1e9:.2f} GB de 15")

sistema = ("Eres un asistente de atención al cliente. Responde en español y solo con "
           "la información del contexto. Si el contexto no alcanza, indícalo.")

def responder(ids, corpus, pregunta, max_new_tokens=160):
    ctx = "\n\n".join(corpus[i] for i in ids)
    msgs = [{"role": "system", "content": #TU CODIGO AQUI},
            {"role": "user", "content": f"{ctx}\n\nPregunta: {#TU CODIGO AQUI}"}]

    e = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                return_dict=True).to(gen.device)
    with torch.no_grad():
        s = gen.generate(**e, max_new_tokens=max_new_tokens, do_sample=False,
                         temperature=None, top_p=None, top_k=None,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(s[0][e["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

t0 = time.time()
print("\n" + responder(recuperados["A_indiceA"][0], CORPUS, consultas[0]))
print(f"({time.time()-t0:.1f}s por respuesta)")

### **i) Dos respuestas que parecen igual de verdad**

Hay dos casos:

* en uno la cruzada pierde el documento correcto. Son las consultas que mueven **recall**
* en el otro la cruzada trae el chunk correcto igual que el baseline pero los otros k-1 son distintos, ahi **recall** no se mueve y **overlap** sí, suelen ser bastantes mas y son las que en producción no se detectan.

In [ ]:
cruzada = "B_indiceA_sinpref"
base_ids = recuperados["A_indiceA"]
cruz_ids = recuperados[cruzada]

pierde el chunk correcto

In [ ]:
perdidas = [j for j in range(len(PARES)) if gold[j] in base_ids[j][:K] and gold[j] not in cruz_ids[j][:K]]

lo trae igual pero con otro contexto alrededor

In [ ]:
silenciosas = [j for j in range(len(PARES))
               if gold[j] in base_ids[j][:K] and gold[j] in cruz_ids[j][:K]
               and set(base_ids[j][:K]) != set(cruz_ids[j][:K])]

de esas, la que más cambió el contexto que es donde la diferencia tiene chance de verse

In [ ]:
silenciosas.sort(key=lambda j: len(set(base_ids[j][:K]) & set(cruz_ids[j][:K])))
print(f"pierde el documento : {len(perdidas)} de {len(PARES)}")
print(f"mismo documento, otro contexto : {len(silenciosas)} de {len(PARES)}")

In [ ]:
def comparar(j, titulo):
    print(f"\n{'='*78}\n{titulo}\nconsulta: {consultas[j]}")
    print(f"esperado: [{gold[j]}] {CORPUS[gold[j]]}\n")
    for etq, ids in (("A_indiceA", base_ids[j][:K]), (cruzada, cruz_ids[j][:K])):
        print(f"recupera {etq:<20} {ids}  {'bien' if gold[j] in ids else 'mal'}")
        for i in ids:
            print(f"   [{i:2d}] {CORPUS[i]}" + ("   <- el que contesta" if i == gold[j] else ""))
    print(f"\nrespuesta con A     : {responder(base_ids[j], CORPUS, consultas[j])}")
    print(f"\nrespuesta con {cruzada}: {responder(cruz_ids[j], CORPUS, consultas[j])}")

if not perdidas and not silenciosas:
    print("\nlas dos condiciones recuperan lo mismo en todas las consultas")

if perdidas:
    comparar(perdidas[0], "caso ruidoso, la cruzada perdió el documento")

if silenciosas:
    comparar(silenciosas[0], "caso silencioso, mismo documento pero otro contexto")

* en el primer caso una respuesta está armada sobre el documento correcto y la otra no
* en el segundo las dos lo tienen pero el resto del contexto cambio y la diferencia puede aparecer o no, las dos están bien escritas

En producción esto no se ve, en los logs no hay error, las respuestas suenan bien, y un test de salida no sirve porque no hay una salida esperada exacta.

### **j) la política contradice al sentido comun**

* La cruzada puede perder el documento y acertar igual, usando lo que el modelo aprendió en el preentrenamiento, eso funciona mientras la política de la empresa coincida con el sentido común

* Un RAG está para responder lo que el modelo no puede saber así que probamos una política que contradiga la intuición

Para que la comparación mida al generador y no al retriever, armamos los dos contextos a mano:
* el que trae la política nueva y el mismo sin ella, esperar a que el retriever pierda ese chunk haría depender la sección de un accidente de recuperación

* como cambia el corpus, reindexamos, si usamos el índice viejo con el corpus nuevo sería otra vez el mismo bug

In [ ]:
pol = 22
corpus2 = list(CORPUS)

escrita sin repetir las palabras de la consulta, para no cambiar sin querer qué tan fácil es de recuperar

In [ ]:
corpus2[pol] = ("La garantía extendida ampara roturas por impacto involuntario durante los primeros 90 días de la compra.")
preg = "se me cayó el producto, ¿la garantía lo cubre?"
marca = "90"
jq = consultas.index(preg)
print("antes :", CORPUS[pol])
print("ahora :", corpus2[pol])

la respuesta correcta cambia y deja de ser la intuitiva

reindexado incremental cambió un chunk y se re-encodea ese solo, vale porque el modelo y su revisión no cambiaron, si hubieran cambiado sería el bug de antes

In [ ]:
indice2 = indice[EMB_A].copy()
indice2[pol] = indexar(EMB_A, [corpus2[pol]])[0]
hash2 = hashlib.sha256("||".join(corpus2).encode()).hexdigest()[:12]
print(f"\nindex_id {iid[EMB_A]} -> {index_id(hash2, EMB_A, revs[EMB_A], CHUNKING)}")

cambió un chunk de 41 y ya es otro artefacto

dos contextos para la misma pregunta: el que trae la política y el mismo sin ella, sacar el chunk a mano en lugar de esperar que el retriever lo pierda deja la comparación bajo control, que es lo que hace falta para medir al generador y no al retriever

In [ ]:
ids_a = buscar(indice2, vecs[(EMB_A, True)][jq])[0]
ids_x = [i for i in buscar(indice2, vecs[(EMB_A, True)][jq], k=K + 1)[0] if i != pol][:K]
if pol not in ids_a:
    print("aviso: el baseline tampoco trae la política, los dos contextos son iguales\n")

for etq, ids in (("con la política", ids_a), ("sin la política", ids_x)):
    print(f"{etq:<18} {ids}")
    for i in ids:
        print(f"   [{i:2d}] {corpus2[i]}" + ("   <- la política real" if i == pol else ""))
    print()

In [ ]:
resp_a = responder(ids_a, corpus2, preg)
resp_x = responder(ids_x, corpus2, preg)
print("respuesta con la política en el contexto\n ", resp_a)
print("\nrespuesta sin la política en el contexto\n ", resp_x)

def cifras_sin_respaldo(respuesta, ids, corpus):
    # proxy barato, si afirma un número que no estaba en el contexto, lo inventó.
    # groundedness en serio necesita un juez, un LLM evaluador o una persona
    ctx = " ".join(corpus[i] for i in ids)
    return sorted(set(re.findall(r"\d+", respuesta)) - set(re.findall(r"\d+", ctx)))

print("\n" + "-" * 66)
for etq, ids, r in (("con la política", ids_a, resp_a), ("sin la política", ids_x, resp_x)):
    print(f"{etq} dice '{marca}': {str(marca.lower() in r.lower()):<6} "
          f"cifras sin respaldo: {cifras_sin_respaldo(r, ids, corpus2) or 'ninguna'}")

Mismo modelo y misma pregunta
* el que recibió el documento contesta bien y el otro contesta mal, con la misma seguridad y la misma redacción
* si antes la cruzada había acertado sin el documento, fue porque la política coincidía con lo que el modelo ya sabía
* cuando la política la contradice, eso deja de funcionar, y deja de funcionar en las preguntas que motivan tener un RAG

Que un RAG acierte, entonces, no dice nada sobre si el retriever anda. El generador puede estar tapando el fallo y lo tapa bien en las preguntas de respuesta obvia

### **k) Hacer auditable el fallo**

Dos defensas para cosas distintas:
* la primera es que el índice no se abra si no corresponde a la configuración pedida, así el error se frena antes de generar una respuesta

In [ ]:
def cargar_indice(npy, meta, corpus_hash, repo, rev, chunking):
    m = json.load(open(meta))
    esperado = index_id(#TU CODIGO AQUI, #TU CODIGO AQUI, #TU CODIGO AQUI, #TU CODIGO AQUI)
    if m["index_id"] != esperado:
        raise RuntimeError(
            f"índice inválido, no corresponde a esta configuración\n"
            f"  guardado: {m['index_id']}  ({m['emb_repo']} @ {m['emb_revision'][:8]}, "
            f"corpus {m['corpus_hash']})\n"
            f"  esperado: {esperado}  ({repo} @ {rev[:8]}, corpus {corpus_hash})\n"
            f"  hay que reindexar el corpus con esta configuración")
    return np.load(npy)

print("1) configuración correcta ->",
      cargar_indice("indice_a.npy", "indice_a.meta.json",
                    corpus_hash, EMB_A, revs[EMB_A], CHUNKING).shape)

print("\n2) modelo cambiado, índice viejo (el bug de la sección e)")
try:
    cargar_indice("indice_a.npy", "indice_a.meta.json", corpus_hash, EMB_B, revs[EMB_B], CHUNKING)
    print("   no detectado, revisar index_id")
except RuntimeError as e:
    print("   detenido antes de generar una sola respuesta\n")
    for linea in str(e).split("\n"):
        print("   " + linea)

print("\n3) corpus cambiado, índice viejo (el bug de la sección j)")
try:
    cargar_indice("indice_a.npy", "indice_a.meta.json", hash2, EMB_A, revs[EMB_A], CHUNKING)
    print("   no detectado")
except RuntimeError:
    print("   detenido")

otro_chunking = {"estrategia": "ventana-deslizante", "size": 500, "overlap": 50}
print("\nlas cuatro patas de la identidad, por separado")
print("original:", iid[EMB_A])
print("un chunk cambiado:", index_id(hash2, EMB_A, revs[EMB_A], CHUNKING))
print("otro chunking:", index_id(corpus_hash, EMB_A, revs[EMB_A], otro_chunking))
print("el modelo B:", index_id(corpus_hash, EMB_B, revs[EMB_B], CHUNKING))

* la segunda defensa es el gate, la identidad no ve el caso en que se cambia el modelo, se reindexa bien, todos los hashes coinciden y el retriever igual anda peor que antes
* esto se detecta comparando contra una medición anterior, que tiene que existir y tiene su configuración al lado, o sea que tiene que estar en MLflow.

* Hay que tener cuidado con qué métrica decide, **overlap** se mide contra el baseline, así que dice cuánto cambió el contexto y no si mejoró.

* Un modelo nuevo que reindexó bien y recupera mejor va a tener overlap bajo porque devuelve otras cosas, y poniéndolo en el gate fallarían también las
mejoras, por eso deciden **recall**, **hit1** y **mrr** contra el gold set, y **overlap** queda como
aviso.

In [ ]:
cliente = MlflowClient()
baseline = "A_indiceA"
cliente.set_tag(runs[baseline], "baseline", "true")   # en un repo esto lo hace el merge a main

tolerancia = {"recall": 0.03, "hit1": 0.05, "mrr": 0.05}
aviso_overlap = 0.80

def gate(nombre):
    base = cliente.get_run(runs[baseline]).data
    cand = cliente.get_run(runs[nombre]).data
    print(f"\n{'='*74}\ngate: {nombre}\n{'='*74}")
    print(f"{'métrica':<12}{'baseline':>10}{'candidato':>12}{'delta':>10}{'tol':>8}   estado")
    ok = True
    for m, tol in tolerancia.items():
        b, c = base.metrics[m], cand.metrics[m]
        pasa = (c - b) >= -tol
        ok = ok and pasa
        print(f"{m:<12}{b:>10.3f}{c:>12.3f}{c-b:>+10.3f}{-tol:>8.2f}   {'pasa' if pasa else 'falla'}")
    if cand.metrics["overlap"] < aviso_overlap:
        print(f"\n  aviso: overlap = {cand.metrics['overlap']:.3f} (< {aviso_overlap}), el contexto")
        print("         cambió mucho, no falla el build pero hay que revisarlo")
    difs = {k: (base.params[k], cand.params[k]) for k in base.params
            if base.params[k] != cand.params.get(k)}
    if difs:
        print("\nparams que cambiaron respecto del baseline:")
        for k, (b, c) in difs.items():
            print(f"  {k}: {b}  ->  {c}")
    print(f"\nresultado: {'ok, se puede promover' if ok else 'falla, no se promueve a producción'}")

gate(cruzada)        # se migró el modelo y no se reindexó
gate("B_indiceB")    # se migró el modelo y sí se reindexó, formalmente impecable

* el primer caso lo habría atajado también la validación de identidad
* el segundo no, porque ahí no hay ningún hash que no coincida y el gate igual tiene algo para decir
* La tolerancia de 0.03 sale de mirar el ancho de los intervalos de la sección f, si el intervalo es de más o menos 0.08, una tolerancia de 0.02 hace fallar builds por ruido de muestreo, y un gate que falla todo el tiempo termina ignorado

In [ ]:
get_ipython().system_raw("mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000 &")
time.sleep(5)
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(5000, height=700)
except ImportError:
    print("fuera de Colab: http://127.0.0.1:5000")